In [3]:
import pandas as pd

from pathlib import Path
import sys

import pandas as pd


def find_project_root(start_path):
    start_path = Path(start_path).resolve()
    for candidate in [start_path, *start_path.parents]:
        if (candidate / 'requirements.txt').exists() and (candidate / 'scripts').exists():
            return candidate
    raise FileNotFoundError('프로젝트 루트 폴더를 찾을 수 없습니다.')


PROJECT_ROOT = find_project_root(Path.cwd())
DATA_DIR = PROJECT_ROOT / 'data' / 'raw'
REPORT_DIR = PROJECT_ROOT / 'reports'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print('Python 실행 파일:', sys.executable)
print('현재 작업 폴더:', Path.cwd())
print('프로젝트 루트:', PROJECT_ROOT)
print('데이터 폴더:', DATA_DIR)
print('결과 저장 폴더:', REPORT_DIR)

required_files = ['customers.csv', 'products.csv', 'orders.csv', 'order_items.csv']
missing_files = [name for name in required_files if not (DATA_DIR / name).exists()]

if missing_files:
    raise FileNotFoundError(
        '필요한 데이터 파일이 없습니다: ' + ', '.join(missing_files)
        + '. 프로젝트 루트에서 python scripts/generate_sample_data.py를 실행하세요.'
    )

customers = pd.read_csv(DATA_DIR / 'customers.csv')
products = pd.read_csv(DATA_DIR / 'products.csv')
orders = pd.read_csv(DATA_DIR / 'orders.csv')
order_items = pd.read_csv(DATA_DIR / 'order_items.csv')

print('데이터 불러오기 완료')

# ─────────────────────────────────────────────
# 1. line_total 생성 (quantity × unit_price)
# ─────────────────────────────────────────────
order_items = order_items.copy()
order_items["line_total"] = order_items["quantity"] * order_items["unit_price"]

# ─────────────────────────────────────────────
# 2. orders.order_id 고유성 확인
#    (many-to-one 병합의 'one' 쪽이 실제로 유일한지 사전 점검)
# ─────────────────────────────────────────────
is_unique = orders["order_id"].is_unique
dup_count = orders["order_id"].duplicated().sum()
print(f"[2] order_id 고유함? {is_unique} / 중복 건수: {dup_count}")

# ─────────────────────────────────────────────
# 4-a. 병합 전 행 수 기록
# ─────────────────────────────────────────────
n_before = len(order_items)

# ─────────────────────────────────────────────
# 3. many-to-one 병합 + indicator
# ─────────────────────────────────────────────
order_sales = order_items.merge(
    orders[["order_id", "customer_id", "order_date", "order_status"]],
    on="order_id",
    how="left",
    validate="many_to_one",   # 왼쪽 다수 : 오른쪽 유일 관계 강제
    indicator=True,           # _merge 열 생성
)

# ─────────────────────────────────────────────
# 4-b. 병합 후 행 수 + 미매칭 확인
# ─────────────────────────────────────────────
n_after = len(order_sales)
merge_counts = order_sales["_merge"].value_counts()
n_unmatched = (order_sales["_merge"] == "left_only").sum()

print(f"[4] 병합 전 행 수: {n_before} / 병합 후 행 수: {n_after} "
      f"/ 행 수 동일? {n_before == n_after}")
print(f"[4] 미매칭(left_only) 건수: {n_unmatched}")
print("[4] _merge 분포:")
print(merge_counts)

# ─────────────────────────────────────────────
# 5. 날짜 변환 + 실패 건수 확인
#    errors='coerce'로 변환 실패는 NaT 처리 후 개수를 셈
# ─────────────────────────────────────────────
order_sales["order_date"] = pd.to_datetime(
    order_sales["order_date"], errors="coerce"
)
n_date_fail = order_sales["order_date"].isna().sum()
print(f"[5] 날짜 변환 실패(NaT) 건수: {n_date_fail}")

# ─────────────────────────────────────────────
# 6. completed 주문만 필터링
# ─────────────────────────────────────────────
completed = order_sales[order_sales["order_status"] == "completed"].copy()
print(f"[6] completed 행 수: {len(completed)}")

# 월별 집계 전, 날짜가 없는 completed 행은 월 구분이 불가하므로 분리
n_completed_no_date = completed["order_date"].isna().sum()
if n_completed_no_date > 0:
    print(f"[6] 주의: completed인데 날짜 없는 행 {n_completed_no_date}건 "
          f"→ 월별 집계에서 제외됨")
completed_dated = completed.dropna(subset=["order_date"])

# ─────────────────────────────────────────────
# 7. 월별 total_sales + 고유 order_count
# ─────────────────────────────────────────────
completed_dated["year_month"] = completed_dated["order_date"].dt.to_period("M")

monthly = (
    completed_dated
    .groupby("year_month")
    .agg(
        total_sales=("line_total", "sum"),
        order_count=("order_id", "nunique"),   # 고유 주문 수
    )
    .reset_index()
)
print("[7] 월별 집계:")
print(monthly)

# ─────────────────────────────────────────────
# 8. 원본 completed total vs 월별 합계 비교 (정합성 교차검증)
# ─────────────────────────────────────────────
total_original = completed["line_total"].sum()          # 날짜 없는 행 포함
total_dated    = completed_dated["line_total"].sum()    # 월별 집계 대상
total_monthly  = monthly["total_sales"].sum()           # 월별 합계의 총합

print(f"[8] completed 전체 합계          : {total_original:,.0f}")
print(f"[8] 월별 집계 대상 합계          : {total_dated:,.0f}")
print(f"[8] 월별 total_sales 총합        : {total_monthly:,.0f}")
print(f"[8] 월별 총합 == 집계대상 합계?   : {total_monthly == total_dated}")
print(f"[8] 날짜 누락으로 인한 차액       : {total_original - total_dated:,.0f}")

# ─────────────────────────────────────────────
# 9. 해석상 주의 (회계 순매출로 단정 금지)
# ─────────────────────────────────────────────
print("[9] 주의: 위 total_sales는 completed 상태 주문의 line_total 합계일 뿐,")
print("         환불·할인·세금·취소분이 반영되지 않았으므로 회계상 순매출로 볼 수 없음.")

Python 실행 파일: c:\dev\llm-data-analysis-course\.venv\Scripts\python.exe
현재 작업 폴더: c:\dev\llm-data-analysis-course\notebooks\ch04
프로젝트 루트: C:\dev\llm-data-analysis-course
데이터 폴더: C:\dev\llm-data-analysis-course\data\raw
결과 저장 폴더: C:\dev\llm-data-analysis-course\reports
데이터 불러오기 완료
[2] order_id 고유함? True / 중복 건수: 0
[4] 병합 전 행 수: 764 / 병합 후 행 수: 764 / 행 수 동일? True
[4] 미매칭(left_only) 건수: 0
[4] _merge 분포:
_merge
both          764
left_only       0
right_only      0
Name: count, dtype: int64
[5] 날짜 변환 실패(NaT) 건수: 0
[6] completed 행 수: 474
[7] 월별 집계:
   year_month  total_sales  order_count
0     2025-09      2650000            5
1     2025-10     16970000           19
2     2025-11     11021000           13
3     2025-12     21085000           23
4     2026-01     14272000           16
5     2026-02      4784000            7
6     2026-03     18323000           25
7     2026-04     11546000           16
8     2026-05     16631000           19
9     2026-06     12504000           17
10    2026-0